# Fixed Income Analytics Service

Run the repo's fixed-income analytics service on one ETF and inspect the snapshot.

In [ ]:
from db.connection import get_engine
from fixed_income.analytics import DurationModelSelector, FixedIncomeAnalyticsService
from fixed_income.instruments.security import Security
from stores.market import PriceStore, MetadataStore
from stores.macro import MacroStore

In [ ]:
DATA_BACKEND = "local"
APP_ENV = "uat"
TICKER = "LQD"
START_DATE = "2024-01-01"

In [ ]:
engine = get_engine(data_backend=DATA_BACKEND, app_env=APP_ENV)
price_store = PriceStore(engine)
metadata_store = MetadataStore(engine)
macro_store = MacroStore(engine)

selector = DurationModelSelector()
analytics_service = FixedIncomeAnalyticsService(price_store, macro_store, selector)

security = Security(
    ticker=TICKER,
    metadata=metadata_store.get_ticker_metadata(TICKER) or {},
    history=price_store.get_ticker_price_history(TICKER, start_date=START_DATE),
)

snapshot = analytics_service.analyze_security(security)
snapshot

In [ ]:
snapshot.to_record()

In [ ]:
print("Ticker:", snapshot.ticker)
print("Bucket:", snapshot.asset_bucket)
print("Benchmark:", snapshot.benchmark_used)
print("Duration:", snapshot.estimated_duration)
print("DV01/share:", snapshot.dv01_per_share)
print("Spread beta/bp:", snapshot.spread_beta_per_bp)
print("Spread proxy:", snapshot.spread_proxy_used)
print("Rate R^2:", snapshot.rate_model_r2)
print("Spread R^2:", snapshot.spread_model_r2)